In [2]:
# Mount drive

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
pip install moviepy


@@@ Metadata for Validation Videos only

In [4]:
import os
from moviepy.editor import VideoFileClip

# Define the directory containing the video files
frame_directory = "/content/drive/MyDrive/work2/Validation_Data/Videos/"

# Initialize variables to store total size and duration
total_size_mb = 0
total_duration_seconds = 0

# Loop through all the files in the directory
for file_name in os.listdir(frame_directory):
    file_path = os.path.join(frame_directory, file_name)

    # Check if the file is a video file (you can add more extensions as needed)
    if file_name.endswith(('.mp4', '.avi', '.mov', '.mkv')):
        # Get the file size in MB
        total_size_mb += os.path.getsize(file_path) / (1024 * 1024)  # Convert bytes to MB

        # Get the duration of the video
        video_clip = VideoFileClip(file_path)
        total_duration_seconds += video_clip.duration
        video_clip.reader.close()

# Output the results
print(f"Total Space of Videos: {total_size_mb:.2f} MB")
print(f"Total Duration of Videos: {total_duration_seconds:.2f} seconds")


Total Space of Videos: 296.66 MB
Total Duration of Videos: 2039.32 seconds


In [ ]:
import cv2
import os
from moviepy.editor import VideoFileClip
import subprocess
import csv

# Function to extract video metadata
def extract_metadata(video_path):
    # Extract resolution, frame rate, and frame count using OpenCV
    cap = cv2.VideoCapture(video_path)
    width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()

    # Extract duration using MoviePy
    clip = VideoFileClip(video_path)
    duration = clip.duration
    clip.close()

    # Get file size
    size = os.path.getsize(video_path) / (1024 * 1024)  # Convert to MB

    # Use FFmpeg to extract codec information
    result = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=codec_name,bit_rate', '-of', 'default=noprint_wrappers=1:nokey=1', video_path], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    codec_info = result.stdout.decode('utf-8').splitlines()

    # Extract video codec and bitrate
    video_codec = codec_info[0] if len(codec_info) > 0 else "Unknown"
    bitrate = codec_info[1] if len(codec_info) > 1 else "Unknown"

    # Use FFmpeg to extract audio codec information
    result_audio = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'a:0', '-show_entries', 'stream=codec_name,sample_rate,channels', '-of', 'default=noprint_wrappers=1:nokey=1', video_path], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    audio_info = result_audio.stdout.decode('utf-8').splitlines()

    # Extract audio codec, sample rate, and channels
    audio_codec = audio_info[0] if len(audio_info) > 0 else "Unknown"
    audio_sample_rate = audio_info[1] if len(audio_info) > 1 else "Unknown"
    audio_channels = audio_info[2] if len(audio_info) > 2 else "Unknown"

    # Calculate aspect ratio
    aspect_ratio = round(width / height, 2)

    return {
        'File': os.path.basename(video_path),
        'Resolution': f'{int(width)}x{int(height)}',
        'Frame Rate': f'{fps:.2f} FPS',
        'Frame Count': int(frame_count),
        'Duration': f'{duration:.2f} seconds',
        'Size': f'{size:.2f} MB',
        'Aspect Ratio': aspect_ratio,
        'Video Codec': video_codec,
        'Bitrate': bitrate,
        'Audio Codec': audio_codec,
        'Audio Sample Rate': audio_sample_rate,
        'Audio Channels': audio_channels
    }

# Directory containing your videos
video_directory = "/content/drive/MyDrive/work2/videos"

# Output CSV file

csv_file = "/content/drive/MyDrive/work2/video_metadata.csv"

# CSV column headers
headers = ['File', 'Resolution', 'Frame Rate', 'Frame Count', 'Duration', 'Size', 'Aspect Ratio', 'Video Codec', 'Bitrate', 'Audio Codec', 'Audio Sample Rate', 'Audio Channels']

# Extract metadata and write to CSV
with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=headers)
    writer.writeheader()  # Write header

    for file_name in os.listdir(video_directory):
        if file_name.endswith(('.mp4', '.mkv', '.avi')):
            video_path = os.path.join(video_directory, file_name)
            metadata = extract_metadata(video_path)
            writer.writerow(metadata)  # Write each video's metadata to the CSV

print(f"Metadata has been saved to {csv_file}")


Metadata has been saved to /content/drive/MyDrive/work2/video_metadata.csv


In [ ]:
import pandas as pd

# Paths to the CSV files
input_csv_file = "/content/drive/MyDrive/work2/video_metadata.csv"
output_csv_file = "/content/drive/MyDrive/work2/video_metadata_Summary.csv"

# Read the original CSV file
df = pd.read_csv(input_csv_file)

# Remove non-numeric parts from 'Duration' and 'Size' columns
df['Duration'] = df['Duration'].str.replace(' seconds', '').astype(float)
df['Size'] = df['Size'].str.replace(' MB', '').astype(float)

# Calculate unique values and metrics
summary = {
    'Resolution': df['Resolution'].unique(),
    'Frame Rate': df['Frame Rate'].unique(),
    'Frame Count': df['Frame Count'].unique(),
    'Minimum Duration': [df['Duration'].min()],
    'Maximum Duration': [df['Duration'].max()],
    'Average Duration': [df['Duration'].mean()],
    'T Duration': [df['Duration'].sum()],
    'Minimum Size': [df['Size'].min()],
    'Maximum Size': [df['Size'].max()],
    'Average Size': [df['Size'].mean()],
    'T Size': [df['Size'].sum()],
    'Aspect Ratios': df['Aspect Ratio'].unique(),
    'Video Codec': df['Video Codec'].unique(),
    'Audio Codec': df['Audio Codec'].unique(),
    'Audio Sample Rate': df['Audio Sample Rate'].unique(),
    'Audio Channels': df['Audio Channels'].unique()
}

# Convert the summary dictionary to a DataFrame
summary_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in summary.items()]))

# Save the summary DataFrame to a new CSV file
summary_df.to_csv(output_csv_file, index=False)

print(f"Summary table has been saved to {output_csv_file}")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/work2/video_metadata.csv'

@@ Dictionary Dataset

In [ ]:
import os
import re

# Function to check if a word is in Urdu
def is_urdu(word):
    # Urdu characters Unicode range is from \u0600 to \u06FF and \u0750 to \u077F
    return bool(re.match(r'^[\u0600-\u06FF\u0750-\u077F]+$', word))

# Function to process the file, filter words, and save the results
def process_file(input_file, output_file1, output_file2):
    # Create the directory if it doesn't exist
    os.makedirs(os.path.dirname(output_file2), exist_ok=True)

    with open(input_file, 'r', encoding='utf-8') as f:
        words = f.readlines()

    # Remove duplicates and filter only Urdu words
    unique_words = set(word.strip() for word in words if is_urdu(word.strip()))

    # Save the filtered words to the output files
    with open(output_file1, 'w', encoding='utf-8') as f_out1, open(output_file2, 'w', encoding='utf-8') as f_out2:
        for word in sorted(unique_words):
            f_out1.write(word + '\n')
            f_out2.write(word + '\n')

# File paths for the output in Google Drive
output_file_urdu_positive_gdrive = "/content/drive/MyDrive/work2/dictionary/urdu_positive_words_NEW.txt"
output_file_urdu_negative_gdrive = "/content/drive/MyDrive/work2/dictionary/urdu_negative_words_NEW.txt"

# Process the positive and negative words files and save in the two required locations
process_file('/content/drive/MyDrive/work2/dictionary/urdu_positive_words_OLD.txt', '/content/drive/MyDrive/work2/dictionary/urdu_positive_filtered.txt', output_file_urdu_positive_gdrive)
process_file('/content/drive/MyDrive/work2/dictionary/urdu_negative_words_OLD.txt', '/content/drive/MyDrive/work2/dictionary/urdu_negative_filtered.txt', output_file_urdu_negative_gdrive)

print("Files have been processed and saved to both local and Google Drive paths.")


Files have been processed and saved to both local and Google Drive paths.


In [ ]:
# Define the list of new Urdu words to append
new_urdu_positive_words = [
     'وفاداری', 'دوستی', 'شرافت', 'عزت', 'ایمانداری', 'خوش اخلاق', 'عافیت', 'خوبصورتی', 'خدمت',
    'محبت', 'احترام', 'اعتماد', 'دیانتداری', 'نیک نیتی', 'خوش بختی', 'حسن', 'عظمت', 'رواداری',
    'دلیری', 'شکریہ', 'فلاح', 'معرفت', 'عزت نفس', 'حلیم', 'خوشی', 'مسرت', 'محبت', 'امن',
    'رحم دلی', 'ہمدردی', 'شفقت', 'قابل', 'امانت', 'محترم', 'ذہانت', 'اعلیٰ', 'کامیابی', 'فیض',
    'سچائی', 'خوبصورت', 'شاندار', 'خوبصورت', 'محفوظ', 'ایمان', 'امید', 'اچھائی', 'بہادر',
    'مددگار', 'محبت بھری', 'مخلص', 'دانشمند', 'روشن خیال', 'قابل احترام', 'با اخلاق', 'سچائی',
    'تعظیم', 'حلیم', 'شفافیت', 'دیانت', 'مثالی', 'ایماندار', 'بلند حوصلہ', 'معزز', 'عزت دار',
    'خوبی', 'صلح', 'امن', 'اعلیٰ ظرف', 'شاندار', 'نرمی', 'معقول', 'مفکر', 'عقلمند', 'صلابت',
    'سکون', 'رحم', 'خوشبو', 'بے غرض', 'روشن', 'بہترین', 'صادق', 'قابل تحسین', 'معزز', 'دوست',
    'نیک دل', 'دانا', 'نیک', 'محفوظ', 'شریف', 'اچھا', 'سچا', 'محترم', 'دلنشین', 'مستحکم',
    'خوشحال', 'پرامن', 'معاون', 'سخی', 'دلچسپ', 'دلفریب', 'نیک نیت', 'حیرت انگیز', 'پیارا',
    'مددگار', 'دلکش', 'محبوب', 'پرامید', 'عقیدت مند', 'خوبصورتی', 'خالص', 'خوشگوار', 'شاندار',
    'ذہین', 'جری', 'حسن سلوک', 'بصیرت', 'فیاض', 'زندگی بخش', 'اچھائی', 'نیکی', 'پیار', 'عظمت',
    'محترم', 'سچائی', 'عظمت', 'خیر', 'بلند مرتبہ', 'سخی', 'پُرخلوص', 'شاندار', 'تسلی بخش', 'قابل اعتماد',
    'بلند ہمت', 'پیشرفت', 'بے خوف', 'محفوظ', 'دلیر', 'خوش قسمت', 'بہادر', 'خوشحال', 'ایماندار', 'پیارا',
    'عزت مآب', 'مہربان', 'حلیم', 'متحمل', 'خوش گفتار', 'بلند ہمت', 'سکون', 'محبت بھر', 'حسن',
    'مطمئن', 'صاف گو', 'فیاضی', 'عزت دار', 'انصاف پسند', 'امن پسند', 'پرسکون', 'متحمل مزاج', 'خوش مزاج',
    'بلند اخلاق', 'قابل تعریف', 'باوقار', 'ایماندار', 'محبت بھرا', 'مطمئن', 'سچائی', 'روشن', 'عزت دار',
    'محفوظ', 'مخلص', 'صبر', 'ہمدردی', 'شفقت', 'خوش گفتار', 'بلند اخلاق', 'پیارا', 'مددگار', 'بے لوث',
    'با ادب', 'نیک دل', 'اعلیٰ', 'بلند حوصلہ', 'قابل فخر', 'مخلص', 'ذہانت', 'خوش طبع', 'معاون', 'خیر خواہ',
    'سچائی', 'امن پسند', 'شائستہ', 'پُر وقار', 'دلنشین', 'رحم دل', 'حلیم', 'سخی', 'خوش بخت', 'بہترین',
    'محترم', 'مہربان', 'دلکش', 'پاکیزہ', 'عالی شان', 'اعلیٰ ظرف', 'بلند مرتبہ', 'حسن اخلاق', 'امانت دار',
    'فہمی', 'سچا', 'قابل اعتماد', 'محفوظ', 'مددگار', 'بے لوث', 'محبت', 'دوستی', 'خوش اخلاق', 'خدمت گزار',
    'ایماندار', 'شفاف', 'عقل مند', 'پیارا', 'ذہین', 'محبت بھر', 'رحم دل', 'مہربان', 'سخی', 'با ادب', 'خوشحال',
    'نرم دل', 'بلند حوصلہ', 'بے خوف', 'خیر خواہ', 'مخلص', 'نیک نیت', 'خوش مزاج', 'قابل تعریف', 'محبوب',
    'خوبصورت', 'نیک دل', 'روشن خیال', 'پیارا', 'خوشبو دار', 'قابل فخر', 'شفاف', 'محفوظ', 'ایماندار',
    'بلند کردار', 'سچائی', 'دلنشین', 'خوشحال', 'بلند حوصلہ', 'پاکیزہ', 'پیار بھرا', 'اچھائی', 'قابل اعتماد',
    'خوبصورت', 'دلنشین', 'بلند ہمت', 'محبت', 'نرم دل', 'مہربان', 'مخلص', 'حسن سلوک', 'فیاض', 'بلند کردار',
    'بہترین', 'بلند اخلاق', 'بے غرض', 'ہمدرد', 'خیر خواہ', 'پیارا', 'سچائی', 'عظمت', 'دوستی', 'مخلص',
    'سخی', 'فیاضی', 'خیر', 'محترم', 'بلند کردار', 'خوشحال', 'بے خوف', 'شائستہ', 'عزت دار', 'محبت', 'عالی شان',
    'پیارا', 'سچا', 'محترم', 'نیک', 'نیک دل', 'شفاف', 'خوبصورت', 'پُرخلوص', 'محبت بھرا', 'سخی', 'شفاف', 'سچائی'
]


# Define the output file path
output_file_urdu_positive_gdrive = "/content/drive/MyDrive/work2/dictionary/urdu_positive_words_NEW.txt"

# Read existing words from the file
existing_words = set()
try:
    with open(output_file_urdu_positive_gdrive, 'r', encoding='utf-8') as file:
        existing_words = set(file.read().splitlines())
except FileNotFoundError:
    # If the file does not exist, we will create it later
    pass

# Prepare a list for unique new words
unique_new_words = [word for word in new_urdu_positive_words if word not in existing_words]

# Append unique new words to the file
if unique_new_words:
    with open(output_file_urdu_positive_gdrive, 'a', encoding='utf-8') as file:
        for word in unique_new_words:
            file.write(word + '\n')

print(f"Successfully appended {len(unique_new_words)} unique Urdu words to {output_file_urdu_positive_gdrive}.")


Successfully appended 144 unique Urdu words to /content/drive/MyDrive/work2/dictionary/urdu_positive_words_NEW.txt.


@@ Audio Dataset

In [ ]:
import os
import subprocess
import csv

# Function to extract audio metadata using FFmpeg
def extract_audio_metadata(audio_path):
    # Use FFmpeg to extract audio codec, sample rate, channels, and duration
    result = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'stream=codec_name,sample_rate,channels',
                             '-select_streams', 'a:0', '-show_entries', 'format=duration',
                             '-of', 'default=noprint_wrappers=1:nokey=1', audio_path],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    audio_info = result.stdout.decode('utf-8').splitlines()

    return {
        'File': os.path.basename(audio_path),
        'Audio Codec': audio_info[0] if len(audio_info) > 0 else "Unknown",
        'Sample Rate': audio_info[1] if len(audio_info) > 1 else "Unknown",
        'Channels': audio_info[2] if len(audio_info) > 2 else "Unknown",
        'Duration': f'{float(audio_info[3]):.2f} seconds' if len(audio_info) > 3 else "Unknown"
    }

# Directory containing your audio files
audio_directory = "/content/drive/MyDrive/work2_extra/wav/"

# Output CSV file for audio metadata
csv_audio_file = "/content/drive/MyDrive/work2/audio_metadata.csv"

# CSV column headers for audio files
audio_headers = ['File', 'Audio Codec', 'Sample Rate', 'Channels', 'Duration']

# Extract audio metadata and write to CSV
with open(csv_audio_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=audio_headers)
    writer.writeheader()  # Write header

    for file_name in os.listdir(audio_directory):
        if file_name.endswith('.wav'):  # You can add more formats if needed
            audio_path = os.path.join(audio_directory, file_name)
            metadata = extract_audio_metadata(audio_path)
            writer.writerow(metadata)  # Write each audio file's metadata to the CSV

print(f"Audio metadata has been saved to {csv_audio_file}")


Audio metadata has been saved to /content/drive/MyDrive/work2/audio_metadata.csv


@@ Text Dataset


In [ ]:
import os
import csv

# Function to extract text file metadata
def extract_text_metadata(text_path):
    with open(text_path, 'r', encoding='utf-8') as file:
        content = file.read()
        lines = content.splitlines()
        words = content.split()

    return {
        'File': os.path.basename(text_path),
        'Number of Words': len(words),
        'Number of Lines': len(lines),
        'Number of Characters': len(content)
    }

# Directory containing your text files
text_directory = "/content/drive/MyDrive/work2/words"

# Output CSV file for text metadata
csv_text_file = "/content/drive/MyDrive/work2/text_metadata.csv"

# CSV column headers for text files
text_headers = ['File', 'Number of Words', 'Number of Lines', 'Number of Characters']

# Extract text metadata and write to CSV
with open(csv_text_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=text_headers)
    writer.writeheader()  # Write header

    for file_name in os.listdir(text_directory):
        if file_name.endswith('.txt'):  # Adjust for your file extensions
            text_path = os.path.join(text_directory, file_name)
            metadata = extract_text_metadata(text_path)
            writer.writerow(metadata)  # Write each text file's metadata to the CSV

print(f"Text metadata has been saved to {csv_text_file}")


Text metadata has been saved to /content/drive/MyDrive/work2/text_metadata.csv


In [ ]:
import os
import csv

# Function to extract text file metadata
def extract_text_metadata(text_path):
    with open(text_path, 'r', encoding='utf-8') as file:
        content = file.read()
        lines = content.splitlines()
        words = content.split()

    return {
        'File': os.path.basename(text_path),
        'Number of Words': len(words),
        'Number of Lines': len(lines),
        'Number of Characters': len(content)
    }

# Directory containing your text files
text_directory = "/content/drive/MyDrive/work2/words"

# Output CSV file for text metadata
csv_text_file = "/content/drive/MyDrive/work2/text_metadata2.csv"

# CSV column headers for text files
text_headers = ['File', 'Number of Words', 'Number of Lines', 'Number of Characters']

# Extract text metadata and write to CSV
with open(csv_text_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=text_headers)
    writer.writeheader()  # Write header

    for file_name in os.listdir(text_directory):
        if file_name.endswith('.txt'):  # Adjust for your file extensions
            text_path = os.path.join(text_directory, file_name)
            metadata = extract_text_metadata(text_path)
            writer.writerow(metadata)  # Write each text file's metadata to the CSV

print(f"Text metadata has been saved to {csv_text_file}")


Text metadata has been saved to /content/drive/MyDrive/work2/text_metadata2.csv


@@ Frames Dataset

In [ ]:
import os
import cv2
import csv

# Function to extract frame metadata
def extract_frame_metadata(frame_path):
    # Load the frame image using OpenCV
    frame = cv2.imread(frame_path)
    height, width, _ = frame.shape


    return {
        'File': os.path.basename(frame_path),
        'Resolution': f'{width}x{height}'
    }

# Directory containing your frame files
frame_directory = "/content/drive/MyDrive/work2/Frames/37/"

# Output CSV file for frame metadata
csv_frame_file = "/content/drive/MyDrive/work2/frame_metadata837.csv"

# CSV column headers for frame files
frame_headers = ['File', 'Resolution']

# Extract frame metadata and write to CSV
with open(csv_frame_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=frame_headers)
    writer.writeheader()  # Write header

    for file_name in os.listdir(frame_directory):
        if file_name.endswith(('.png', '.jpg', '.jpeg')):  # Adjust for your image formats
            frame_path = os.path.join(frame_directory, file_name)
            metadata = extract_frame_metadata(frame_path)
            writer.writerow(metadata)  # Write each frame file's metadata to the CSV

print(f"Frame metadata has been saved to {csv_frame_file}")


Frame metadata has been saved to /content/drive/MyDrive/work2/frame_metadata837.csv


@@@@


In [ ]:
import os

# Function to calculate number of files, folders, and total size of a directory
def calculate_directory_stats(directory_path):
    total_size = 0
    file_count = 0
    folder_count = 0

    for dirpath, dirnames, filenames in os.walk(directory_path):
        # Count the number of folders
        folder_count += len(dirnames)

        # Count the number of files and calculate their total size
        file_count += len(filenames)
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)

    return folder_count, file_count, total_size / (1024 * 1024)  # Convert size to MB

# Define the directories
audio_directory = "/content/drive/MyDrive/work2_extra/wav/"
text_directory = "/content/drive/MyDrive/work2/words"
frames_directory = "/content/drive/MyDrive/work2/Frames/"

# Get statistics for the audio directory
audio_folders, audio_files, audio_size_mb = calculate_directory_stats(audio_directory)
print(f"Audio Directory:\nFolders: {audio_folders}, Files: {audio_files}, Size: {audio_size_mb:.2f} MB")

# Get statistics for the text directory
text_folders, text_files, text_size_mb = calculate_directory_stats(text_directory)
print(f"Text Directory:\nFolders: {text_folders}, Files: {text_files}, Size: {text_size_mb:.2f} MB")

# Get statistics for the frames directory
frames_folders, frames_files, frames_size_mb = calculate_directory_stats(frames_directory)
print(f"Frames Directory:\nFolders: {frames_folders}, Files: {frames_files}, Size: {frames_size_mb:.2f} MB")


Audio Directory:
Folders: 0, Files: 111, Size: 544.68 MB
Text Directory:
Folders: 0, Files: 112, Size: 0.16 MB
Frames Directory:
Folders: 111, Files: 3137, Size: 284.00 MB
